# **Object Counting Logic**
---
I used this notebook to build and analyse the object counting layer on top of detection results. Counting answers: 'How many of each class appear in each frame?' and 'How does the count change over the duration of the video?'

In [1]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import sys, json
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

from detection_system.counting import (
    count_objects,
    count_objects_over_time,
    summarise_counts,
    get_dominant_classes,
)
from detection_system.utils import load_config, save_figure, save_table

cfg = load_config("config.yaml")
plt.style.use(cfg["figures"]["style"])
TOP_N = cfg["counting"]["top_n_classes"]
print(f"Top N classes to highlight: {TOP_N}")
print("Setup complete")

Top N classes to highlight: 5
Setup complete


In [ ]:
# ── Loading detection results ────────────────────────────────────────────
JSON_PATH = PROJECT_ROOT / "data" / "processed" / "detections.json"

if not JSON_PATH.exists():
    raise FileNotFoundError(
        f"{JSON_PATH} not found.\n"
        "Run Notebook 04 first to generate detection results."
    )

with open(JSON_PATH, "r") as f:
    data = json.load(f)

all_detections = data["frame_detections"]
fps_video = data["fps_video"]
frame_count = data["frame_count"]

print(f"Loaded detections from: {JSON_PATH.name}")
print(f"  Frames           : {frame_count}")
print(f"  Video FPS        : {fps_video:.1f}")
print(f"  Video duration   : {frame_count / fps_video:.1f} s")

# Count total detections across all frames
total_detections = sum(len(f) for f in all_detections)
print(f"  Total detections : {total_detections}")
print(f"  Mean per frame   : {total_detections / frame_count:.2f}")

Loaded detections from: detections.json
  Frames           : 1501
  Video FPS        : 25.0
  Video duration   : 60.0 s
  Total detections : 10725
  Mean per frame   : 7.15


In [ ]:
# ── Per-frame count for one frame (sanity check) ──────────────────────
# I verified count_objects() on a single frame before applying it to all frames.

frame_0_dets = all_detections[0]
frame_0_counts = count_objects(frame_0_dets)

print(f"Frame 0 — {len(frame_0_dets)} detections:")
if frame_0_counts:
    for cls, cnt in sorted(frame_0_counts.items(), key=lambda x: -x[1]):
        print(f"  {cls:<16}: {cnt}")
else:
    print("  No detections in frame 0.")
    print("  (Expected for mock video — synthetic frames have no real objects)")

Frame 0 — 9 detections:
  car             : 7
  truck           : 2


In [ ]:
# ── Building the full count time-series ──────────────────────────────────

count_df = count_objects_over_time(all_detections)

print(f"Count DataFrame shape: {count_df.shape}")
print(f"  Rows (frames)    : {count_df.shape[0]}")
print(f"  Columns (classes): {count_df.shape[1]}")
print(f"  Classes detected : {list(count_df.columns)}")

# Add a time column (seconds) for plotting
count_df["time_s"] = count_df.index / fps_video

print(f"\nFirst 5 frames:")
display(count_df.head())

Count DataFrame shape: (1501, 5)
  Rows (frames)    : 1501
  Columns (classes): 5
  Classes detected : ['bus', 'car', 'motorcycle', 'person', 'truck']

First 5 frames:


,bus,car,motorcycle,person,truck,time_s
frame,,,,,,
0,0,7,0,0,2,0.00
1,0,8,0,1,2,0.04
2,0,9,0,0,1,0.08
3,0,8,0,0,1,0.12
4,0,8,0,0,3,0.16


In [ ]:
# ── Summary statistics per class ─────────────────────────────────────
# This computes totals, means, peaks, and presence rates across all frames for each detected class.

summary = summarise_counts(count_df.drop(columns=["time_s"], errors="ignore"))

print("Count summary (sorted by total appearances):")
display(summary)

# Save to CSV and LaTeX
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
count_df.to_csv(PROCESSED_DIR / "count_timeseries.csv")
summary.to_csv(PROCESSED_DIR / "count_summary.csv")
save_table(summary, "05_count_summary")

print(f"\n✓ Count time-series saved to data/processed/count_timeseries.csv")
print(f"✓ Count summary saved to data/processed/count_summary.csv")

Count summary (sorted by total appearances):


,total_appearances,mean_per_frame,max_in_single_frame,frames_present,pct_frames_present
class_name,,,,,
car,7776,5.18,9,1501,100.0
truck,2044,1.36,5,1241,82.7
person,745,0.50,5,468,31.2
motorcycle,155,0.10,2,150,10.0
bus,5,0.00,1,5,0.3


  Saved: reports\tables\05_count_summary.csv
  Saved: reports\tables\05_count_summary.tex
  Saved: paper\tables\05_count_summary.csv
  Saved: paper\tables\05_count_summary.tex

✓ Count time-series saved to data/processed/count_timeseries.csv
✓ Count summary saved to data/processed/count_summary.csv


In [ ]:
# ── Count time-series plot ────────────────────────────────────────────

class_cols = [c for c in count_df.columns if c != "time_s"]
top_classes = get_dominant_classes(
    count_df[class_cols], top_n=min(TOP_N, len(class_cols))
)

if top_classes:
    fig, ax = plt.subplots(figsize=(14, 5))
    
    palette = sns.color_palette("colorblind", n_colors=len(top_classes))
    
    for cls, color in zip(top_classes, palette):
        ax.plot(
            count_df["time_s"],
            count_df[cls],
            label=cls,
            color=color,
            alpha=0.85,
            linewidth=1.5,
        )
    
    ax.set_xlabel("Time (seconds)", fontsize=12)
    ax.set_ylabel("Objects detected per frame", fontsize=12)
    ax.set_title(
        f"Object Count Over Time — Top {len(top_classes)} Classes (YOLOv8-small)",
        fontsize=13, fontweight="bold"
    )
    ax.legend(title="Class", fontsize=10)
    ax.grid(alpha=0.4)
    plt.tight_layout()
    
    save_figure(fig, "05_count_timeseries")
    plt.show()
else:
    print("No detected classes to plot. (Expected for mock video.)")
    print("Generate a figure placeholder...")
    
    # Placeholder figure for mock video
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.text(0.5, 0.5, "No real detections — using mock video\n"
            "Run with a real video to see count time-series",
            ha="center", va="center", fontsize=14, transform=ax.transAxes,
            color="grey")
    ax.set_title("Object Count Over Time — No Detections (Mock Data)", fontsize=13)
    save_figure(fig, "05_count_timeseries")
    plt.show()

  Saved: reports\figures\05_count_timeseries.png
  Saved: reports\figures\05_count_timeseries.pdf
  Saved: paper\figures\05_count_timeseries.png
  Saved: paper\figures\05_count_timeseries.pdf


In [ ]:
# ── Per-class total count bar chart ───────────────────────────────────

if not summary.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    
    colors = sns.color_palette("colorblind", n_colors=len(summary))
    bars = ax.bar(
        summary.index,
        summary["total_appearances"],
        color=colors,
        edgecolor="white",
    )
    
    for bar, val in zip(bars, summary["total_appearances"]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                str(val), ha="center", va="bottom", fontsize=9)
    
    ax.set_xlabel("Object class", fontsize=12)
    ax.set_ylabel("Total detections (all frames)", fontsize=12)
    ax.set_title("Total Detection Count per Class", fontsize=13, fontweight="bold")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    
    save_figure(fig, "05_count_summary_bar")
    plt.show()
else:
    print("No detections in any frame — skipping bar chart.")

  Saved: reports\figures\05_count_summary_bar.png
  Saved: reports\figures\05_count_summary_bar.pdf
  Saved: paper\figures\05_count_summary_bar.png
  Saved: paper\figures\05_count_summary_bar.pdf


In [ ]:
# ── Counting stability check ─────────────────────────────────────────
# I checked frame-to-frame variation in counts for the top class.
# High variation suggests the model is inconsistent (objects appear/disappear across adjacent frames), which is a known limitation of frame-level counting without temporal tracking.

if top_classes:
    primary_class = top_classes[0]
    primary_counts = count_df[primary_class]
    
    # Frame-to-frame difference
    diffs = primary_counts.diff().dropna().abs()
    
    print(f"Counting stability for '{primary_class}':")
    print(f"  Mean count per frame          : {primary_counts.mean():.2f}")
    print(f"  Std of count per frame        : {primary_counts.std():.2f}")
    print(f"  Mean |frame-to-frame change|  : {diffs.mean():.2f}")
    print(f"  Max  |frame-to-frame change|  : {diffs.max():.0f}")
    print()
    print("Interpretation:")
    print("  A high mean |frame-to-frame change| indicates counting instability.")
    print("  This is expected with frame-level counting (no temporal tracking).")
    print("  Temporal tracking (e.g. SORT, ByteTrack) would reduce this variance.")
    print("  This limitation is acknowledged in the technical report.")
else:
    print("No classes detected — stability check skipped.")

Counting stability for 'car':
  Mean count per frame          : 5.18
  Std of count per frame        : 1.46
  Mean |frame-to-frame change|  : 0.18
  Max  |frame-to-frame change|  : 2

Interpretation:
  A high mean |frame-to-frame change| indicates counting instability.
  This is expected with frame-level counting (no temporal tracking).
  Temporal tracking (e.g. SORT, ByteTrack) would reduce this variance.
  This limitation is acknowledged in the technical report.
